<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/10_coding_llm_cli.ipynb)


# colab — Coding LLM on the Command Line

## _Serving a Code Model on Colab GPU and Using It from a CLI Agent_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Download a code-focused LLM, serve it with an OpenAI-compatible
    API, tunnel the endpoint out of Colab, and use it from the command line
    with curl, the OpenAI Python client, and a real coding agent (aider).
- **Hardware**: Designed for Google Colab (T4 default, L4 optional). A T4
    (16 GB VRAM) is sufficient for the default quantised 7B model.
- **Approach**: Use Ollama, which avoids slow source builds on Colab, serves
    an OpenAI-compatible API, and runs `Qwen2.5-Coder` locally on the GPU.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU
    comparison table.

### Roadmap

1. **Environment**: Check the GPU and install dependencies.
2. **Model server**: Start Ollama and pull the Qwen2.5-Coder model.
3. **API check**: Verify the OpenAI-compatible endpoint.
4. **Tunnel**: Expose the server with `cloudflared` to get a public URL.
5. **CLI with curl**: Send requests directly from the command line.
6. **CLI with OpenAI Python client**: Programmatic access from Python.
7. **CLI with aider**: Full coding agent pointed at the served model.
8. **Cleanup**: Stop the server and free resources.

### 1 — Environment

Verify that a GPU is available and install the packages needed for serving,
tunnelling, and the coding agent.

In [ ]:
#@title Check GPU
!nvidia-smi

In [ ]:
#@title Install Ollama, cloudflared, and client tools
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/\
latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm cloudflared-linux-amd64.deb
!pip -q install aider-chat openai
!ollama --version
!cloudflared --version  # verify install

### 2 — Start Ollama and Pull the Model

Ollama ships prebuilt binaries, avoiding the slow `llama-cpp-python` source
build on Colab. We start the local Ollama server and pull
`qwen2.5-coder:7b`, a quantised coding model that fits on a T4.

In [ ]:
#@title Start Ollama server and pull Qwen2.5-Coder 7B
import subprocess
import time
import urllib.request

PORT = 11434
MODEL_NAME = "qwen2.5-coder:7b"

def ollama_ready():
    try:
        urllib.request.urlopen(
            f"http://localhost:{PORT}/api/tags", timeout=2
        )
        return True
    except Exception:
        return False

if not ollama_ready():
    server_proc = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print(f"Ollama server PID: {server_proc.pid}")
    for _ in range(60):
        if ollama_ready():
            print("Ollama server is ready.")
            break
        time.sleep(2)
    else:
        raise RuntimeError("Ollama server did not start in time.")
else:
    server_proc = None
    print("Ollama server is already running.")

!ollama pull {MODEL_NAME}

### 3 — Verify the API Server

Ollama exposes a local API on port `11434`, including an OpenAI-compatible
`/v1` endpoint. The next cell verifies that the model is available through
that endpoint.

In [ ]:
#@title Verify the OpenAI-compatible API endpoint
import json
import urllib.request

with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models") as r:
    models = json.load(r)

print(json.dumps(models, indent=2))

### 4 — Tunnel with Cloudflare

Colab does not expose ports to the internet. We use `cloudflared` to create
a free, temporary public URL that forwards to our local server. No account
is needed. The URL changes each session and typically lasts as long as the
Colab runtime.

In [ ]:
#@title Start cloudflared tunnel and capture the public URL
import re

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url",
     f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("Waiting for tunnel URL ...")

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stderr.readline().decode(
        errors="replace"
    )
    m = re.search(
        r"(https://[a-z0-9\-]+\.trycloudflare\.com)", line
    )
    if m:
        tunnel_url = m.group(1)
        break

if tunnel_url:
    print(f"Public URL: {tunnel_url}")
    print(f"API base:   {tunnel_url}/v1")
else:
    print("Could not get tunnel URL. Rerun this cell.")

### 5 — CLI with curl

The simplest way to interact with the model from any terminal. The server
implements the OpenAI chat-completions endpoint, so the request format is
identical to calling the OpenAI API.

In [ ]:
#@title curl: list available models
import subprocess
import json

if tunnel_url:
    out = subprocess.check_output(
        ["curl", "-s", f"{tunnel_url}/v1/models"]
    )
    print(json.dumps(json.loads(out), indent=2))
else:
    print("No tunnel URL available. Run the tunnel cell first.")

In [ ]:
#@title curl: chat completion request
import json
import subprocess

if tunnel_url:
    payload = json.dumps({
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": "Write a "
            "Python function that computes the nth "
            "Fibonacci number using memoisation."},
        ],
        "max_tokens": 256,
    })
    cmd = ["curl", "-s",
           f"{tunnel_url}/v1/chat/completions",
           "-H", "Content-Type: application/json",
           "-d", payload]
    out = subprocess.check_output(cmd, text=True)
    print(json.dumps(json.loads(out), indent=2))
else:
    print("No tunnel URL available.")

### 6 — CLI with OpenAI Python Client

For programmatic access, the `openai` Python library can point at any
OpenAI-compatible endpoint. This is how most agent frameworks and tools
connect to self-hosted models.

In [ ]:
#@title OpenAI client: chat completion
from openai import OpenAI
import textwrap

if not tunnel_url:
    raise RuntimeError("No tunnel URL. Run the tunnel cell first.")

client = OpenAI(
    base_url=f"{tunnel_url}/v1",
    api_key="dummy",  # any non-empty string works
)

resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are an expert Python "
         "programmer. Keep answers concise."},
        {"role": "user", "content": "Write a Python "
         "decorator that retries a function on exception."},
    ],
    max_tokens=300,
    temperature=0.3,
)

print(textwrap.fill(
    resp.choices[0].message.content, width=80
))

In [ ]:
#@title OpenAI client: streaming response
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user","content":"Explain list comprehension "
         "in Python with two short examples."},
    ],
    max_tokens=200,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

### 7 — CLI with aider (Coding Agent)

[aider](https://aider.chat) is a full-featured AI coding agent that works
from the terminal. It can edit files, run shell commands, and apply code
changes autonomously. Because aider requires an interactive TTY, we
demonstrate two modes:

- **Non-interactive (`--message`)**: Works directly in notebook cells.
    You pass a single instruction; aider edits files and returns.
- **Interactive (Colab Terminal)**: Open the Colab Terminal (left sidebar
    icon) for a real shell where you can run aider interactively with
    full back-and-forth conversation.

In both cases, aider connects to our served model via the tunnel URL.

In [ ]:
#@title Configure aider for the served model
import os

if not tunnel_url:
    raise RuntimeError("No tunnel URL. Run the tunnel cell first.")

os.environ["OPENAI_API_BASE"] = f"{tunnel_url}/v1"
os.environ["OPENAI_API_KEY"] = "dummy"
AIDER_MODEL = f"openai/{MODEL_NAME}"
print(f"OPENAI_API_BASE = {tunnel_url}/v1")
print(f"AIDER_MODEL     = {AIDER_MODEL}")

In [ ]:
#@title Create a small Python project for aider to work on
import os

PROJECT_DIR = "/content/aider_demo"
os.makedirs(PROJECT_DIR, exist_ok=True)

with open(os.path.join(PROJECT_DIR, "calc.py"), "w") as f:
    f.write(
        "def add(a, b):\n"
        "    return a + b\n"
        "\n"
        "def multiply(a, b):\n"
        "    return a * b\n"
    )

print("Created", os.path.join(PROJECT_DIR, "calc.py"))
print(open(os.path.join(PROJECT_DIR, "calc.py")).read())

In [ ]:
#@title aider: non-interactive mode (edits files, returns)
message = (
    "Add a subtract and a divide function to calc.py. "
    "Add a docstring to each function. Include a __main__ "
    "block that runs a quick sanity check."
)

cmd = [
    "python", "-m", "aider",
    "--yes-always",
    "--no-auto-commits",
    "--model", AIDER_MODEL,
    "--message", message,
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=False)

In [ ]:
#@title Show the updated file
result = open(os.path.join(PROJECT_DIR, "calc.py")).read()
print(result)

#### Interactive aider via Colab Terminal

For a full interactive coding session (back-and-forth chat, file editing,
multi-turn refinement), use the **Colab Terminal**:

1. Click the **Terminal** icon in the left sidebar of Colab.
2. Set the environment variables and launch aider:

```bash
export OPENAI_API_BASE=<your-tunnel-url>/v1
export OPENAI_API_KEY=dummy
cd /content/aider_demo
python -m aider --model openai/qwen2.5-coder:7b
```

Replace `<your-tunnel-url>` with the URL printed by the tunnel cell above.
You can then chat with aider, ask it to add features, refactor, fix bugs,
and it will edit your files in place.

### Throughput Benchmark

A quick benchmark to measure tokens per second on the served model. This
helps set expectations: a Q4_K_M 7B model on T4 typically delivers
10–15 tok/s, which is usable for coding but slower than a hosted API.

In [ ]:
#@title Measure tokens per second via the API
import time

prompt = "Write a Python function that merges two sorted lists."

start = time.time()
resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128,
    temperature=0.0,
)
elapsed = time.time() - start

content = resp.choices[0].message.content
tok_count = resp.usage.completion_tokens
print(f"Prompt:    {prompt}")
print(f"Tokens:    {tok_count}")
print(f"Time:      {elapsed:.1f} s")
print(f"Throughput: {tok_count / elapsed:.1f} tok/s")
print(f"\nOutput:\n{textwrap.fill(content, width=80)}")

### Swapping Models

Other Ollama models that can work on a T4:

| Model | Size | Notes |
|-------|------|-------|
| `qwen2.5-coder:7b` | 7 B | Default in this notebook |
| `qwen2.5-coder:3b` | 3 B | Faster, less capable |
| `qwen2.5-coder:1.5b` | 1.5 B | Very fast, basic coding |
| `qwen2.5:7b` | 7 B | General-purpose, not code-specific |

To swap, change `MODEL_NAME` in the Ollama pull cell. For larger models,
use an L4 or A100 Colab runtime.

### 8 — Cleanup

Stop the server and tunnel processes to free GPU memory and Colab resources.

In [ ]:
#@title Stop server and tunnel
if server_proc is not None:
    server_proc.terminate()
tunnel_proc.terminate()
print("Server and tunnel stopped.")

### Exercises

1. **Smaller model**: Swap in `qwen2.5-coder:3b` and compare throughput
     and code quality on the same prompt.
2. **Streaming curl**: Use `curl --no-buffer` with `"stream": true` in the
     request body and observe tokens arriving incrementally.
3. **aider multi-step**: Use aider `--message` to first add a feature, then
     run it again on the same project to add unit tests.
4. **Alternative tunnel**: Replace `cloudflared` with Colab's built-in
     `google.colab.output.eval_js(f"google.colab.kernel.proxyPort({PORT})")`
     and note the trade-offs (private URL, no account needed).
5. **Larger GPU**: Try a larger coding model on an L4 or A100 runtime and
     compare speed, memory use, and answer quality.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
